##### Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 FunctionGemma 的完整函數呼叫序列

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/functiongemma/full-function-calling-sequence-with-functiongemma"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/docs/functiongemma/full-function-calling-sequence-with-functiongemma.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/docs/functiongemma/full-function-calling-sequence-with-functiongemma.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemini%2Fgemma-cookbook%2Fmain%2Fdocs%2Ffunctiongemma%2Ffull-function-calling-sequence-with-functiongemma.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemini/gemma-cookbook/blob/main/docs/functiongemma/full-function-calling-sequence-with-functiongemma.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

FunctionGemma 是 Gemma 3 270M 模型的專門版本，專門針對函數呼叫改進進行了訓練。它具有與Gemma相同的架構，但使用不同的聊天格式和tokenizer。

本指南展示了在Hugging Face生態系中使用FunctionGemma的完整工作流程。它涵蓋了基本的設定步驟，包括安裝必要的Python包，例如`torch`和`transformers`，以及透過Hugging Face Hub載入模型。本教學的核心演示了將模型連接到外部工具的三階段循環：**模型的轉向**生成函數調用對象，**開發人員的轉向**解析和執行代碼（例如天氣API），以及**最終響應**，其中模型使用工具的輸出來回答用戶。

## 設定

在開始本教學之前，請完成以下步驟：
* 透過登入 [Hugging Face](https://huggingface.co/google/functiongemma-270m-it) 並選擇 FunctionGemma 模型的 **確認許可證** 來存取 FunctionGemma。
* 產生Hugging Face [存取權杖](https://huggingface.co/docs/hub/en/security-tokens#how-to-manage-user-access-token) 並將其新增至您的Colab 環境。

這個notebook將在 CPU 或 GPU 上執行。

## 安裝 Python 軟體包

安裝執行 FunctionGemma 模型和發出請求所需的 Hugging Face 庫。

In [ ]:
# Install PyTorch & other libraries
!pip install torch

# Install the transformers library
!pip install transformers

接受許可證後，您需要有效的 Hugging Face token 才能存取模型。

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login
login()

## 負載模型

使用`torch` 和`transformers` 庫透過`AutoProcessor` 和`AutoModelForCausalLM` 類別建立`processor` 和`model` 的實例，如下列程式碼範例所示：

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

GEMMA_MODEL_ID = "google/functiongemma-270m-it"

processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(GEMMA_MODEL_ID, dtype="auto", device_map="auto")

## 範例用例

函數呼叫將Gemma的生成能力與外部資料和服務連接起來。以下是一些常見的應用：
* **用即時數據回答問題：** 使用搜尋引擎或天氣API來回答諸如「東京的天氣怎麼樣？」之類的問題。或“誰贏得了最新的 F1 比賽？”
* **控制外部系統：** 將 Gemma 連接到其他應用程式以執行操作，例如發送電子郵件（「向團隊發送有關會議的提醒」）、管理日曆或控制智慧家庭設備。
* **建立複雜的工作流程**：將多個工具呼叫連結在一起以完成多步驟任務，例如透過尋找航班、預訂飯店和建立日曆活動來規劃旅行。

## 使用工具

函數呼叫的核心涉及四步驟過程：
1.  **定義工具**：建立模型可以使用的函數，指定參數和描述（例如天氣查找函數）。
2.  **輪到模型**：FunctionGemma 接收使用者的prompt 和可用工具清單。它產生一個特殊的對象，指示要呼叫哪個函數以及使用什麼參數，而不是純文字回應。
3.  **輪到開發人員了**：您的程式碼接收此對象，使用提供的參數執行指定的函數，並格式化要傳回模型的結果。
4.  **最終回應**：FunctionGemma 使用函數的輸出產生最終的、面向使用者的回應。

讓我們模擬一下這個過程。

In [ ]:
# Define a function that our model can use.
def get_current_weather(location: str, unit: str = "celsius"):
    """
    Gets the current weather in a given location.

    Args:
        location: The city and state, e.g. "San Francisco, CA" or "Tokyo, JP"
        unit: The unit to return the temperature in. (choices: ["celsius", "fahrenheit"])

    Returns:
        temperature: The current temperature in the given location
        weather: The current weather in the given location
    """
    return {"temperature": 15, "weather": "sunny"}


### 模特兒的回合

這是使用者 prompt `"Hey, what's the weather in Tokyo right now?"` 和工具 `[get_current_weather]`。 FunctionGemma 生成函數呼叫物件如下。

In [ ]:
prompt = "Hey, what's the weather in Tokyo right now?"
tools = [get_current_weather]

message = [
        # ESSENTIAL SYSTEM PROMPT:
        # This line activates the model's function calling logic.
        {"role": "developer", "content": "You are a model that can do function calling with the following functions"},
        {"role": "user", "content": prompt},
]

inputs = processor.apply_chat_template(message, tools=tools, add_generation_prompt=True, return_dict=True, return_tensors="pt")
output = processor.decode(inputs["input_ids"][0], skip_special_tokens=False)

out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
generated_tokens = out[0][len(inputs["input_ids"][0]):]
output = processor.decode(generated_tokens, skip_special_tokens=True)

print(f"Prompt: {prompt}")
print(f"Tools: {tools}")
print(f"Output: {output}")

Prompt: Hey, what's the weather in Tokyo right now?
Tools: [<function get_current_weather at 0x79b7e0f52e80>]
Output: <start_function_call>call:get_current_weather{location:<escape>Tokyo, Japan<escape>}<end_function_call>


> 注意：為了確保 FunctionGemma 正確解釋可用工具並產生結構化呼叫而不是純文本，**開發人員**訊息至關重要。這個特定的 system prompt 指示模型它具有執行函數呼叫的權限和能力。

```python
message = [
        # ESSENTIAL SYSTEM PROMPT:
        # This line activates the model's function calling logic.
        {"role": "developer", "content": "You are a model that can do function calling with the following functions"},
        {"role": "user", "content": prompt},
]
```

### 輪到開發商了

您的應用程式應該解析模型的回應以提取函數名稱和參數，並使用 `tool` 角色附加函數呼叫結果。
> 注意：在執行之前始終驗證函數名稱和參數。

In [ ]:
import re

def extract_tool_calls(text):
    def cast(v):
        try: return int(v)
        except:
            try: return float(v)
            except: return {'true': True, 'false': False}.get(v.lower(), v.strip("'\""))

    return [{
        "name": name,
        "arguments": {
            k: cast((v1 or v2).strip())
            for k, v1, v2 in re.findall(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", args)
        }
    } for name, args in re.findall(r"<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>", text, re.DOTALL)]

calls = extract_tool_calls(output)
if calls:
    message.append({
        "role": "assistant",
        "tool_calls": [{"type": "function", "function": call} for call in calls]
    })
    print(message[-1])

    # Call the function and get the result
    #####################################
    # WARNING: This is a demonstration. #
    #####################################
    # Using globals() to call functions dynamically can be dangerous in
    # production. In a real application, you should implement a secure way to
    # map function names to actual function calls, such as a predefined
    # dictionary of allowed tools and their implementations.
    results = [
        {"name": c['name'], "response": globals()[c['name']](**c['arguments'])}
        for c in calls
    ]

    message.append({
        "role": "tool",
        "content": results
    })
    print(message[-1])


{'role': 'assistant', 'tool_calls': [{'type': 'function', 'function': {'name': 'get_current_weather', 'arguments': {'location': 'Tokyo, Japan'}}}]}
{'role': 'tool', 'content': [{'name': 'get_current_weather', 'response': {'temperature': 15, 'weather': 'sunny'}}]}


> 注意：為了獲得最佳結果，請使用以下特定格式將工具執行結果附加到您的訊息記錄中。這可確保聊天範本正確產生所需的 token 結構（例如，`response:get_current_weather{temperature:15,weather:<escape>sunny<escape>}`）。

```python
message.append({
    "role": "tool",
    "content": {
        "name": function_name,
        "response": function_response
    }
})
```

如果有多個獨立請求：
```python
message.append({
    "role": "tool",
    "content": [
        {
            "name": function_name_1,
            "response": function_response_1
        },
        {
            "name": function_name_2,
            "response": function_response_2
        }
    ]
})
```


### 最終回應

最後，FunctionGemma讀取工​​具回應並回覆給使用者。

In [ ]:
inputs = processor.apply_chat_template(message, tools=tools, add_generation_prompt=True, return_dict=True, return_tensors="pt")
out = model.generate(**inputs.to(model.device), pad_token_id=processor.eos_token_id, max_new_tokens=128)
generated_tokens = out[0][len(inputs["input_ids"][0]):]
output = processor.decode(generated_tokens, skip_special_tokens=True)
print(f"Output: {output}")
message.append({"role": "assistant", "content": output})

Output: The current weather in Tokyo is sunny with a temperature of 15 degrees Celsius.


您可以在下面查看完整的聊天記錄。

In [ ]:
# full history
for item in message:
  print(item)

print("-"*80)
output = processor.decode(out[0], skip_special_tokens=False)
print(f"Output: {output}")

{'role': 'developer', 'content': 'You are a model that can do function calling with the following functions'}
{'role': 'user', 'content': "Hey, what's the weather in Tokyo right now?"}
{'role': 'assistant', 'tool_calls': [{'type': 'function', 'function': {'name': 'get_current_weather', 'arguments': {'location': 'Tokyo, Japan'}}}]}
{'role': 'tool', 'content': [{'name': 'get_current_weather', 'response': {'temperature': 15, 'weather': 'sunny'}}]}
{'role': 'assistant', 'content': 'The current weather in Tokyo is sunny with a temperature of 15 degrees Celsius.'}
--------------------------------------------------------------------------------
Output: <bos><start_of_turn>developer
You are a model that can do function calling with the following functions<start_function_declaration>declaration:get_current_weather{description:<escape>Gets the current weather in a given location.<escape>,parameters:{properties:{location:{description:<escape>The city and state, e.g. "San Francisco, CA" or "Tokyo,

## 摘要與後續步驟

您已經了解如何建立可以使用 FunctionGemma 呼叫函數的應用程式。工作流程是透過四個階段循環建立的：
1.  **定義工具**：建立模型可以使用的函數，指定參數和描述（例如天氣查找函數）。
2.  **模型輪到**：模型接收使用者的prompt和可用工具列表，傳回結構化函數呼叫物件而不是純文字。
3.  **輪到開發人員了**：開發人員使用正規表示式解析此輸出以提取函數名稱和參數，執行實際的 Python 程式碼，並使用特定工具角色將結果附加到聊天記錄中。
4. **最終回應**：模型處理工具的執行結果，為使用者產生最終的自然語言答案。

查看以下文件以進一步閱讀。
- [使用FunctionGemma微調](https://ai.google.dev/gemma/docs/functiongemma/finetuning-with-functiongemma)
